# Traceprop-LLM -- Stage 2: speed sweep (Pythia-1B) + matched-proj_dim Traceprop overhead + LDS sweep (from-scratch) + item 9 rerun + backdoor detection

Supersedes the earlier single-backend design. Five steps now run on the settings where each actually produces signal, per compute-cost analysis (see below):

1. **Speed + measured storage** (`exp31`) stays on **Pythia-1B**, tracked scope `{last-1, last-6, all}` -- this is where overhead differences actually show up, and it's the number that matters for the paper's systems claim.
2. **Traceprop's OWN overhead at the LogIX-matched proj_dim** (`exp25`, Step 1b) -- `exp31` only reports the `proj_dim` Traceprop would need to match LogIX's real footprint; it never times Traceprop itself at that budget. This closes that gap, one `exp25` run per scope at the measured-matching `proj_dim`.
3. **LDS / compression-quality** (`exp35`) moves to the **from-scratch tiny LoRA transformer** (`--backend tiny`), tracked scope `{last-1, all}`. Pythia-1B/SST-2 is a LoRA fine-tune of an already-pretrained model on a task it partly solves -- the paper's own §3.4 data shows LDS near zero there for every method, so a LogIX-vs-Traceprop LDS comparison in that regime would likely just be noise, and it's expensive. The tiny/synthetic setting has real signal (LDS ~0.65-0.70, confirmed in `results/exp27_tiny_tiny-clf.json`) and is CPU-fast.
4. **Item 9 rerun** (`exp29`) -- the number already published in §3.4 (inline vs. final-checkpoint LDS, GPT-2/SST-2), rerun at the paper's existing settings (n_subsets=500) so the raw per-test-example npz exists for a paired bootstrap (the currently-published run predates that code).
5. **Backdoor detection** (`exp37`, PRIMARY) stays on **Pythia-1B**, ~1% overhead setting. Hardened across four rounds of review, most recently: **`auc_poison_vs_distractor` is a label-sign artifact, not attribution evidence** -- a per-example output-layer gradient's sign follows the TRAINING label (poison=target label, distractor=non-target label), so that AUC hit 1.0 in an earlier smoke test where `backdoor_success_rate` was still 0, i.e. before the model had learned anything about the trigger. It's now kept ONLY as a sign-sanity check (renamed `auc_poison_vs_distractor_SIGN_SANITY_CHECK_ONLY` in the output). The PRIMARY metric is now **`auc_within_target`**: AUC computed only among training examples labeled with the target class (poisoned vs. naturally target-labeled clean examples) -- every row here shares the same label, so a `label_match` baseline is exactly 0.5 by construction and any real separation has to come from the trigger. `label_match` and `label_aware_repr` (hidden-state cosine x label agreement) are reported as fair, label-aware baselines for every AUC -- if `label_aware_repr` matches or beats gradient methods, that's a real possible outcome and gets reported honestly, not as a failure. The script itself now flags a seed's backdoor AUCs invalid (`backdoor_learned: false`) when the triggered-minus-clean gap is below `--min_backdoor_gap` (default 0.3), and the aggregated numbers exclude invalid seeds automatically. Mislabel detection via self-influence is kept as a SECONDARY result. Raw per-example arrays (every score variant, labels, is_backdoor/is_distractor/is_mislabel, target rates) are saved per seed in the npz, so any metric can be recomputed later without another GPU run.

**A SEPARATE notebook, a separate Colab session**: `exp34_table2_sweep_colab.ipynb` -- Table 2 (head-to-head post-hoc-vs-inline speedup) clean rerun, plus Traceprop's own general tracked-parameters sweep (NOT storage-matched to LogIX -- that's Step 1b above, a different, newer ask). Still pending in the paper, unrelated to this notebook's LogIX comparison; run it whenever, doesn't need to be the same session as this one.

**Framing note for the paper**: `exp35`'s LDS comparison scores BOTH tools on post-hoc, final-checkpoint gradients (isolates the compression scheme, not an inline-quality claim). Inline-quality evidence comes from item 9 (§3.4) and the backdoor experiment, both Traceprop-only -- already stated in Related Work and the §3.3 pending-marker comment in `main.tex`. **Be ready for `label_aware_repr` to match gradient attribution on `auc_within_target`** -- if that happens, the honest headline is "inline attribution recovers poisoned examples as well as strong label-aware baselines, at ~1% overhead, from logs collected during training," not "gradients beat representations." Where gradients clearly have the edge is LDS / counterfactual prediction, which `exp35` covers -- know which claim the numbers support before writing them up.

**Compute estimate** (using real measured step times: Pythia-1B 424ms/step, GPT-2 96ms/step, both at batch=16/seq=64, from `results/exp25_hf_pythia-1b_track1.json` / `exp25_hf_gpt2_track1.json`):
- Step 1 (exp31, speed): full settings only at track=1, reduced at track={6,0}: **~2.8 hours**.
- Step 1b (exp25, Traceprop at matched proj_dim): same reduced-settings pattern, one config per scope instead of two: **~1.4 hours**.
- Step 2 (exp35, LDS, tiny backend, CPU): **~1 hour**.
- Step 3 (exp29, item 9 rerun, n_subsets=500 matching the paper's existing methodology, GPT-2): **~2.5 hours** -- the single biggest addition this round; if the session is running long, this is the one step that could reasonably move to its own separate short session later (the paper already has this exact number, only the raw npz is new).
- Step 4 (exp37, backdoor, 5 seeds): **~20-30 min**.
- **New total: roughly 8-8.5 hours.** Substantially more than the ~4.5h estimated before Step 1b and Step 3 were added -- results write to Drive (`/content/drive/MyDrive/traceprop_runs/`) after each step, so a disconnect only costs whatever step was mid-run, and Step 3 in particular can be deferred to a second session if needed without blocking the rest.

## Setup: pin versions, mount Drive, assert L4, clone repo

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert 'L4' in torch.cuda.get_device_name(0), (
    f"expected an L4, got {torch.cuda.get_device_name(0)} -- Table 1/2, the sweep, and this "
    f"session's runs are all L4-only; a different GPU would make the numbers incomparable. "
    f"Reconnect and request an L4 runtime."
)

# The old pins (transformers==4.44.2, peft==0.13.2, accelerate==0.34.2,
# "numpy<2") were chosen on an earlier Colab image (Python 3.10/3.11, numpy
# 1.x-era torch). Colab is now on Python 3.13 with numpy 2.x as the default,
# self-consistent with jax/opencv/ml-dtypes/etc already in the base image --
# both the old pins AND "numpy<2" are now actively harmful: transformers/peft
# can't even build (no Python 3.13 wheel for the old tokenizers version) and
# downgrading numpy breaks the ABI every other preinstalled package expects
# (ValueError: numpy.dtype size changed -- classic numpy 1.x/2.x C-extension
# mismatch). Fix: install nothing that fights Colab's current numpy, drop the
# unbuildable transformers/peft/accelerate pins (confirmed Colab's current
# defaults work with our code up to the torchao check below), and just remove
# torchao, whose presence (Colab preinstalls 0.10.0) makes newer peft's LoRA
# dispatch raise on a version check even though nothing here uses it.
!pip -q install datasets scipy scikit-learn accelerate
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install --ignore-requires-python logix-ai

import numpy, transformers, peft
print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)

Mounted at /content/drive


In [3]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .
%cd /content/Traceprop/experiments

GitHub token: ··········
/content/Traceprop
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done
/content/Traceprop/experiments


## Step 0: 1-repeat GPU dry run (do this before anything else)

Only exp31 and exp37 touch the GPU (Pythia-1B); exp35 moved to the CPU-fast tiny backend and was already validated locally, no GPU dry run needed for it. Gradient validation stays ON. If either of these fails, fix it here -- minute 5, not hour 2.

In [4]:
!python exp31_logix_comparison.py --backend hf --model EleutherAI/pythia-1b --device cuda \
    --steps 5 --repeats 1 --warmup 1 --pca_cov_steps 2 --track 1 \
    --out /tmp/dryrun_exp31.json --force

config.json: 100% 569/569 [00:00<00:00, 2.99MB/s]
tokenizer_config.json: 100% 396/396 [00:00<00:00, 2.23MB/s]
tokenizer.json: 100% 2.11M/2.11M [00:00<00:00, 130MB/s]
special_tokens_map.json: 100% 99.0/99.0 [00:00<00:00, 602kB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/2.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:  11% 226M/2.09G [00:01<00:06, 267MB/s, 14.8MB/s  ]
model.safetensors: downloading bytes:  15% 319M/2.09G [00:02<00:05, 331MB/s, 25.8MB/s  ]
model.safetensors: downloading bytes:  22% 467M/2.09G [00:02<00:03, 407MB/s, 37.5MB/s  ]
model.safetensors: downloading bytes:  29% 614M/2.09G [00:02<00:02, 546MB/s, 47.5MB/s  ]
model.safetensors: downloading bytes:  38% 785M/2.09G [00:02<00:02, 548MB/s, 65.7MB/s  ]
model.safetensors: reconstructing file:  29% 604M/2.09G [00:02<00:04, 324MB/s, 42.8MB/s  ]
model.safetensors: downloading bytes:  52% 1.09G/2.09G [00:03<00:01, 619MB/s, 88.5MB/s  ]
model.saf

In [5]:
!python exp37_planted_detection.py --backend hf --data sst2 --model EleutherAI/pythia-1b --device cuda \
    --n_train 32 --n_trigger_test 16 --plant_frac 0.15 --distractor_frac 0.1 --mislabel_frac 0.1 \
    --epochs 1 --batch 8 --track 1 --n_seeds 1 \
    --out /tmp/dryrun_exp37.json --force

[exp37] === seed 0 (1/1) ===
README.md: 100% 35.3k/35.3k [00:00<00:00, 71.7MB/s]

sst2/train-00000-of-00001.parquet: downloading bytes:  87% 2.71M/3.11M [00:01<00:00, 1.91MB/s]
sst2/train-00000-of-00001.parquet: downloading bytes: 100% 2.99M/2.99M [00:01<00:00, 1.86MB/s,  283kB/s  ]
sst2/train-00000-of-00001.parquet: reconstructing file: 100% 3.11M/3.11M [00:01<00:00, 1.93MB/s,  298kB/s  ]

sst2/validation-00000-of-00001.parquet: downloading bytes:   0% 0.00/72.8k [00:00<?, ?B/s]
sst2/validation-00000-of-00001.parquet: downloading bytes: 100% 71.1k/71.1k [00:01<00:00, 65.7kB/s, 6.91kB/s  ]
sst2/validation-00000-of-00001.parquet: reconstructing file: 100% 72.8k/72.8k [00:01<00:00, 67.2kB/s, 7.08kB/s  ]

sst2/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/148k [00:00<?, ?B/s]
sst2/test-00000-of-00001.parquet: downloading bytes: 100% 145k/145k [00:01<00:00, 143kB/s, 14.1kB/s  ]
sst2/test-00000-of-00001.parquet: reconstructing file: 100% 148k/148k [00:01<00:00, 145kB/s, 14.4kB/s

**If both printed `gradient validation OK` (exp31) and finished without error, proceed. If anything failed, stop and fix it before running the real sweep below.**

## Step 1: exp31 speed sweep on Pythia-1B -- full settings at track=1, reduced at track={6,0}

Full settings (steps=200, repeats=20) at track=1 match Table 1's existing methodology exactly. track={6,0} use steps=100/repeats=10 (secondary sweep points, less statistical power but ~4x cheaper) -- raise back to full settings if there's time budget left after everything else finishes.

In [6]:
sweep_settings = [
    (1, 'last1', 200, 20),
    (6, 'last6', 100, 10),
    (0, 'all',   100, 10),
]
for track, label, steps, repeats in sweep_settings:
    print(f'=== exp31 track={track} ({label}), steps={steps} repeats={repeats} ===')
    !python exp31_logix_comparison.py --backend hf --model EleutherAI/pythia-1b --device cuda \
        --steps {steps} --repeats {repeats} --warmup 10 --pca_cov_steps 20 --track {track} \
        --out results/exp31_pythia1b_track{track}.json --force
    !cp results/exp31_pythia1b_track{track}.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

=== exp31 track=1 (last1), steps=200 repeats=20 ===
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 196/196 [00:00<00:00, 679.84it/s]
Traceback (most recent call last):
  File "/content/Traceprop/experiments/exp31_logix_comparison.py", line 467, in <module>
    main()
    ~~~~^^
  File "/content/Traceprop/experiments/exp31_logix_comparison.py", line 463, in main
    run(args)
    ~~~^^^^^^
  File "/content/Traceprop/experiments/exp31_logix_comparison.py", line 104, in run
    _template = build_model()
  File "/content/Traceprop/experiments/exp31_logix_comparison.py", line 100, in build_model
    return build_hf_model(args.model, r=args.rank).to(device)
           ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Traceprop/experiments/exp25_llm_inline_overhead.py", line 129, in build_hf_model
    model = get_peft_model(base, cfg)
  File "/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py", line 198, in get_peft_model
    return MOD

## Step 1b: Traceprop's OWN overhead at the LogIX-matched proj_dim, per scope

exp31 only measures LogIX's speed and reports the `proj_dim` Traceprop would need to match LogIX's real on-disk footprint at each scope -- it never times Traceprop itself at that budget. This runs `exp25` (Traceprop's own overhead script) at each scope's `traceprop_proj_dim_to_match`, so the paper can report BOTH tools' overhead at the same storage budget, not just LogIX's. Same reduced-settings pattern as Step 1 (full at track=1, lighter at track={6,0}).

In [7]:
import json

for track, label, steps, repeats in sweep_settings:
    d = json.load(open(f'results/exp31_pythia1b_track{track}.json'))
    matched_proj_dim = d['storage_matching']['traceprop_proj_dim_to_match']
    print(f'=== exp25 (Traceprop) track={track} ({label}), proj_dim={matched_proj_dim} '
          f'(matched to LogIX), steps={steps} repeats={repeats} ===')
    !python exp25_llm_inline_overhead.py --backend hf --model EleutherAI/pythia-1b --device cuda \
        --steps {steps} --repeats {repeats} --warmup 10 --track {track} --proj_dim {matched_proj_dim} \
        --out results/exp25_pythia1b_track{track}_matched{matched_proj_dim}.json --force
    !cp results/exp25_pythia1b_track{track}_matched{matched_proj_dim}.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

FileNotFoundError: [Errno 2] No such file or directory: 'results/exp31_pythia1b_track1.json'

## Step 2: exp35 LDS sweep on the from-scratch tiny LoRA transformer -- {last-1, all}

CPU is fine here (fast regardless); n_subsets=200 (lower-powered than the paper's canonical 500 for §3.4, raise if there's time). n_blocks=2 (default) means track=0 ("all") already covers both blocks, so track={1,0} is the full 2-point sweep this model supports.

In [ ]:
for track, label in [(1, 'last1'), (0, 'all')]:
    print(f'=== exp35 (tiny backend) track={track} ({label}) ===')
    !python exp35_logix_lds.py --backend tiny --device cpu \
        --n_train 400 --n_test 100 --n_subsets 200 --subset_frac 0.5 --epochs 3 \
        --track {track} --lora_init pca \
        --out results/exp35_tiny_track{track}.json --force
    !cp results/exp35_tiny_track{track}.json results/exp35_tiny_track{track}_raw.npz \
        /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Step 3: item 9 rerun (exp29) -- inline vs. final-checkpoint LDS, real settings, raw scores saved

This is the number already in the paper's §3.4 (inline dot/TRAK vs. final-checkpoint dot/TRAK on GPT-2/SST-2). `exp29`'s code already saves raw per-test-example Spearman-r arrays plus the masks/margins matrices to a `*_raw.npz` file (needed for a paired bootstrap over test examples, without another GPU session) -- the version currently in the paper predates that code and has no such npz. Rerunning at the paper's existing settings (n_subsets=500) gets both a fresh number (should land close to what's already published, same seed) and the raw data the bootstrap needs.

In [ ]:
!python exp29_inline_vs_final_lds.py --backend hf --data sst2 --model gpt2 --device cuda \
    --n_train 1000 --n_test 200 --n_subsets 500 --subset_frac 0.5 --epochs 3 \
    --proj_dim 256 --track 1 \
    --out results/exp29_hf_gpt2_rerun.json --force
!cp results/exp29_hf_gpt2_rerun.json results/exp29_hf_gpt2_rerun_raw.npz \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Step 4: backdoor detection (exp37, PRIMARY) on Pythia-1B, ~1% overhead setting

Mislabel detection (self-influence vs. loss-ranking vs. gradient-norm baselines) runs alongside as the secondary result, same training pass.

In [ ]:
!python exp37_planted_detection.py --backend hf --data sst2 --model EleutherAI/pythia-1b --device cuda \
    --n_train 2000 --n_trigger_test 200 --plant_frac 0.05 --distractor_frac 0.02 --mislabel_frac 0.05 \
    --epochs 3 --batch 16 --proj_dim 512 --track 1 --n_seeds 5 \
    --out results/exp37_pythia1b_track1.json --force
!cp results/exp37_pythia1b_track1.json results/exp37_pythia1b_track1_raw.npz \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Read everything back

In [ ]:
import json

print('=== exp31: LogIX overhead + measured bytes, by tracked scope (Pythia-1B) ===')
matched_proj_dims = {}
for track, label, *_ in sweep_settings:
    d = json.load(open(f'results/exp31_pythia1b_track{track}.json'))
    sm = d['storage_matching']
    matched_proj_dims[track] = sm['traceprop_proj_dim_to_match']
    print(f"  track={label}: tracked_modules={d['tracked_modules_count']} "
          f"measured_bytes/ex={sm['logix_bytes_per_example_measured']} "
          f"matching_proj_dim={sm['traceprop_proj_dim_to_match']}")
    for cfg_name, cfg in d['configs'].items():
        print(f"    {cfg_name}: overhead={cfg['overhead_pct_median']}% +/- {cfg['overhead_pct_std']}%"
              + (f" covariance_pass={cfg['covariance_pass_s']}s" if cfg.get('covariance_pass_s') else ''))
    gv = d['gradient_validation']
    print(f"    gradient_validation: worst_cosine={min(g['worst_cosine'] for g in gv):.6f} "
          f"worst_nonuniformity={max(g['scale_nonuniformity'] for g in gv):.6f}")

print('\n=== exp25: Traceprop\'s OWN overhead at the LogIX-matched proj_dim, by tracked scope ===')
for track, label, *_ in sweep_settings:
    mpd = matched_proj_dims[track]
    d = json.load(open(f'results/exp25_pythia1b_track{track}_matched{mpd}.json'))
    print(f"  track={label} (proj_dim={mpd}): "
          f"throughput_overhead={d['throughput_overhead_pct']}% +/- {d['throughput_overhead_std']}%")

print('\n=== exp35: LDS, by tracked scope (from-scratch tiny transformer) ===')
for track, label in [(1, 'last1'), (0, 'all')]:
    d = json.load(open(f'results/exp35_tiny_track{track}.json'))
    sm = d['storage_matching']
    print(f"  track={label}: measured_bytes/ex={sm['logix_bytes_per_example_measured']} "
          f"matched_proj_dim={sm['traceprop_proj_dim_matched']}")
    for k, v in d['lds'].items():
        print(f"    {k:<22} {v['mean']:+.4f} +/- {v['std']:.4f}")
    gv = d['gradient_validation']
    print(f"    gradient_validation: worst_cosine={gv['worst_cosine']:.6f} "
          f"nonuniformity={gv['scale_nonuniformity']:.6f}")

print('\n=== exp29: item 9 rerun -- inline vs. final-checkpoint LDS (GPT-2/SST-2) ===')
d = json.load(open('results/exp29_hf_gpt2_rerun.json'))
for k, v in d['lds'].items():
    print(f"    {k:<14} {v['mean']:+.4f} +/- {v['std']:.4f}")
print(f"    (compare to the already-published exp29_hf_gpt2.json numbers -- should land "
      f"close, same seed/settings; raw npz now available for the paired bootstrap)")

print('\n=== exp37: backdoor detection (primary) + mislabel detection (secondary), Pythia-1B, 5 seeds ===')
d = json.load(open('results/exp37_pythia1b_track1.json'))

def fmt(m):
    return f"{m['mean']:.4f} +/- {m['std']:.4f}"

# Check the backdoor actually took BEFORE trusting any AUC below -- a small or
# negative gap means the model didn't really learn the trigger. The script
# itself already excluded any such seed from the aggregates below (see
# n_valid_seeds / per_seed_backdoor_learned), but surface it here too.
success = d['backdoor_success_rate']['mean']
clean = d['clean_target_rate']['mean']
gap = success - clean
print(f"  *** backdoor gap (triggered - clean target rate): {gap:+.4f} "
      f"({success:.4f} - {clean:.4f}) ***")
print(f"  *** valid seeds: {d['n_valid_seeds']}/{d['n_seeds_total']} "
      f"(per-seed: {d['per_seed_backdoor_learned']}) ***")
if d['n_valid_seeds'] == 0:
    print(f"  *** WARNING: NO seed reached the backdoor gap threshold -- the backdoor AUCs "
          f"below fell back to using ALL seeds anyway and should not be trusted. Raise "
          f"--plant_frac or --epochs and rerun. ***")
elif d['n_valid_seeds'] < d['n_seeds_total']:
    print(f"  *** NOTE: {d['n_seeds_total'] - d['n_valid_seeds']} seed(s) excluded from the "
          f"backdoor aggregates below for failing the gap threshold. ***")

print(f"\n  inline overhead: {fmt(d['overhead_pct'])}%")
print(f"  random-score AUC baseline: {fmt(d['backdoor_random_auc'])}")
ovf, ovf_tot = d['trigger_overflow_rows_total'], d['trigger_overflow_denominator_total']
print(f"  trigger overflow (had to overwrite content, no pad room to append): "
      f"{ovf}/{ovf_tot} rows ({100*ovf/ovf_tot:.1f}%) -- should be low for SST-2 at seq=32/64")
print(f"\n  PRIMARY backdoor attribution (k={d['k_backdoor']}, distractors={d['k_distractor']}):")
print(f"  auc_within_target is the number that matters -- label_match should be ~0.5 there")
print(f"  by construction; sign_sanity_check is NOT attribution evidence (label-sign artifact).")
for name, v in d['primary_backdoor'].items():
    print(f"    {name:<18} auc_within_target={fmt(v['auc_within_target'])}  "
          f"sign_sanity_check={fmt(v['auc_poison_vs_distractor_SIGN_SANITY_CHECK_ONLY'])}")
print(f"\n  SECONDARY mislabel detection (k={d['k_mislabel']}, vs. loss/grad_norm baselines):")
for name, v in d['secondary_mislabel'].items():
    print(f"    {name:<16} AUC={fmt(v['auc'])}  precision@k={fmt(v['precision_at_k'])}")

## Bring results back

Paste the printed summary above to update the paper draft: (1) fill in the §3.3 storage-matched draft from `docs/mlsys/LOGIX_OUTCOME_DRAFTS.md` using the real numbers, (2) add the scope-sweep table to `main.tex` (fills in the currently-pending "overhead vs. tracked parameters" claim, using exp31's Pythia-1B data), (3) add the LDS-vs-scope table using exp35's tiny-backend data, explicitly framed as a final-checkpoint compression-quality comparison, (4) write up the backdoor detection result as the paper's headline attribution-works evidence, with mislabel detection plus the loss/grad-norm baselines as a secondary table.